# VectorBT Quickstart for MFT

This notebook is tuned for a denser mid-frequency workflow using real BTC/ETH market data.

What it does:
1. Download real BTC/ETH 5-minute data from Yahoo Finance
2. Run a simple moving-average crossover backtest
3. Split one asset into train / validation / test rolling windows
4. Evaluate parameter stability across splits to reduce overfitting risk

Why this setup:
- `5min` is a much better fit than `15min` if you want denser MFT-style observations
- Yahoo intraday data is still limited, so this notebook stretches to the latest 60 days
- Rolling validation is a better first filter than a single in-sample backtest

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt

print('vectorbt version:', vbt.__version__)

In [ ]:
# Yahoo Finance uses interval strings like '5m'.
# vectorbt portfolio stats still use a pandas-style frequency like '5min'.
freq = '5min'
interval = '5m'

data = vbt.YFData.download(
    ['BTC-USD', 'ETH-USD'],
    period='60d',
    interval=interval,
    missing_index='drop'
)

price = data.get('Close')
print('price shape:', price.shape)
price.tail()

In [ ]:
# 48 and 192 bars on 5-minute data are roughly 4-hour and 16-hour moving averages.
fast_ma = vbt.MA.run(price, 48)
slow_ma = vbt.MA.run(price, 192)

entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

entries.tail()

In [ ]:
pf = vbt.Portfolio.from_signals(
    price,
    entries,
    exits,
    init_cash=1000,
    fees=0.001,
    freq=freq
)

pf.stats()

In [ ]:
pf.total_return()

In [ ]:
# pf.plot() expects one fully selected portfolio column.
# Since this notebook runs multiple symbols and parameter combinations,
# plotting portfolio value is the most stable overview.
pf.value().vbt.plot().show()

## Walk-forward validation

A single good backtest can be overfit.

Below we use BTC and split it into rolling windows:
- train: 35 days
- validation: 10 days
- test: 10 days

Then we evaluate multiple MA parameter combinations on each split.

Interpretation:
- If train is great but validation and test collapse, the strategy is likely fragile
- If performance is at least directionally consistent across splits, that is a better sign

In [ ]:
# Use BTC here to keep the validation step fast and easy to read.
wf_price = price['BTC-USD']

split_kwargs = dict(
    n=4,
    window_len=55 * 24 * 12,
    set_lens=(35 * 24 * 12, 10 * 24 * 12),
    left_to_right=True
)

(train_price, train_indexes), (val_price, val_indexes), (test_price, test_indexes) = wf_price.vbt.rolling_split(**split_kwargs)

print('train shape:', train_price.shape)
print('validation shape:', val_price.shape)
print('test shape:', test_price.shape)

In [ ]:
# A small grid keeps the notebook responsive while still showing the workflow.
# These windows span roughly 2 hours to 16 hours on 5-minute bars.
windows = np.arange(24, 193, 24)

def simulate_all_params(price):
    fast, slow = vbt.MA.run_combs(price, windows, r=2, short_names=['fast', 'slow'])
    entries = fast.ma_crossed_above(slow)
    exits = fast.ma_crossed_below(slow)
    pf = vbt.Portfolio.from_signals(
        price,
        entries,
        exits,
        init_cash=1000,
        fees=0.001,
        freq=freq
    )
    return pf.sharpe_ratio()

train_sharpe = simulate_all_params(train_price)
val_sharpe = simulate_all_params(val_price)
test_sharpe = simulate_all_params(test_price)

print('train grid size:', len(train_sharpe))
train_sharpe.head()

In [ ]:
# Pick the best parameter set on train only, then inspect validation and test.
best_idx = train_sharpe[train_sharpe.groupby('split_idx').idxmax()].index

summary = pd.DataFrame({
    'train_sharpe': train_sharpe.loc[best_idx].values,
    'validation_sharpe': val_sharpe.loc[best_idx].values,
    'test_sharpe': test_sharpe.loc[best_idx].values,
}, index=best_idx).reset_index()

summary['generalization_gap'] = summary['train_sharpe'] - summary['validation_sharpe']
summary

In [ ]:
summary[['train_sharpe', 'validation_sharpe', 'test_sharpe', 'generalization_gap']].describe()